### Setup

In [ ]:
# from brmspy import brms
import pandas as pd
import geopandas as gpd
import cartopy
import numpy as np

In [ ]:
# brms setup - this *should* autodetect all the necessary libraries.
# with brms.manage() as ctx:
#     ctx.install_brms(use_prebuilt=True)

### Summarize phenological transitions

In [ ]:
meta = pd.read_parquet("phenocam_site_in_usa.parquet")
pheno = pd.read_parquet("phenocam_site_in_usa_phenology.parquet")

In [ ]:
GCC_FIELD = "gcc_90" # 90th percentile GCC within each image

pheno_clean = pheno.query(f"gcc_value == '{GCC_FIELD}'")
pheno_clean = pheno_clean.assign(
    # Relativize greenenss transitions for slope calculation
    rel_green_50=lambda x: np.minimum(1, (x.threshold_50 - x.min_gcc) / (x.max_gcc - x.min_gcc)),
    rel_green_10=lambda x: np.maximum(0, (x.threshold_10 - x.min_gcc) / (x.max_gcc - x.min_gcc)),
)

# Convert date columns to date dtype
date_columns = list(filter(lambda x: x.startswith("transition_"), pheno_clean.columns))

pheno_clean[date_columns] = pheno_clean[date_columns].apply(pd.to_datetime)

# Determine transition year
pheno_clean = pheno_clean.assign(
    transition_year = lambda x: np.where(
        x.direction == "falling", 
        np.minimum(x.transition_50.dt.year, x.transition_10.dt.year),
        np.maximum(x.transition_50.dt.year, x.transition_10.dt.year)
    )
)

pheno_clean["transition_year_start"] = pd.to_datetime(pheno_clean["transition_year"].map(lambda x: f"{x}-01-01"))

# Get DOY of transition dates relative to the transition year
yday_columns = list(map(lambda x: "doy_" + x, date_columns))

for col in date_columns:
    new_col = "doy_"+col
    pheno_clean[new_col] = (pheno_clean[col] - pheno_clean.transition_year_start).dt.days

# Calculate transition slope
pheno_clean = pheno_clean.assign(
    transition_slope = lambda x: np.abs((np.log(2 * x.rel_green_50 / x.rel_green_10) - 1) / (x.doy_transition_50 - x.doy_transition_10))
)

# Set index keys for joining
# pheno_clean = pheno_clean.set_index(["site", "veg_type", "roi_id"])

In [ ]:
pheno_clean

### Pair rising/falling transitions

In [ ]:
pheno_rising  = pheno_clean.query("direction == 'rising'").sort_values(by="transition_50").rename(columns={"transition_50":"transition_50_rising"})
pheno_falling = pheno_clean.query("direction == 'falling'").sort_values(by="transition_50").rename(columns={"transition_50":"transition_50_falling"})

In [ ]:
pheno_join = pd.merge_asof(
    pheno_rising,
    pheno_falling,
    by=["site", "roi_id", "veg_type"],
    left_on="transition_50_rising",
    right_on="transition_50_falling",
    direction="forward",
    suffixes=("_rising", "_falling")
).drop(columns=["direction_rising", "direction_falling"])

pheno_join = pheno_join.assign(
    season_length = lambda x: (x.transition_50_falling - x.transition_50_rising).dt.days
)

# Shift rising transitions after yday 180 backward 365 days for consistency.
# It shouldn't matter for the circular regression, but it is useful for
# visualizations.
yday_shift = np.where(pheno_join["doy_transition_50_rising"] > 180, -365, 0)
doy_columns = pheno_join.filter(regex="doy.*rising").columns
pheno_join[doy_columns] = pheno_join[doy_columns].apply(lambda x: x + yday_shift)

In [ ]:
pheno_join.hist("doy_transition_50_rising", by="veg_type", sharex=True)

### Join climatology/weather

Two transformations:
 - Long-term normal
 - Rolling 3-year anomaly

In [ ]:
site_climate_raw = pd.read_parquet("phenocam_site_in_usa_climate.parquet")
site_climate_raw.columns = ["year", "yday", "prcp", "tmax", "tmin", "vp", "site"]
data_columns = ["prcp", "tmax", "tmin", "vp"]

site_climate_ann = site_climate_raw.groupby(["site", "year"])\
    .apply(lambda x: x.assign(prcp_tmax_corr=lambda y: np.corrcoef(y.prcp, y.tmax)[0, 1]))\
    .drop(columns=["site", "year", "yday"])\
    .groupby(["site", "year"])\
    .agg("mean")

In [ ]:
for dc in data_columns:
    site_climate_ann[dc+"_lt"] = site_climate_ann.groupby("site")[dc].transform("mean")
    site_climate_ann[dc+"_roll"] = site_climate_ann.groupby("site")[dc].transform(
        lambda x: x.rolling(window=3).mean()
    )
    site_climate_ann[dc+"_anomaly"] = site_climate_ann[dc+"_roll"] - site_climate_ann[dc+"_lt"]

In [ ]:
site_climate_pheno = pd.merge(
    pheno_join,
    site_climate_ann,
    left_on=["site", "transition_year_rising"],
    right_on=["site", "year"]
)

# Attach the remapped veg types
site_climate_pheno["veg_type"] = site_climate_pheno["site"].map(meta["primary_veg_type"])

# Transform response variable and clamp to [-pi, pi]
site_climate_pheno["rad_transition_50_rising"] = np.clip(
    (site_climate_pheno["doy_transition_50_rising"] * 2 * np.pi / 365),
    -np.pi,
     np.pi
)

### Fit models

First: start of season DOY. We want to embed scaling in the model so that we do not have to do transforms ourselves.

## Transition slopes

There doesn't seem to be much relationship between transition slope and climate. Just use the mean transition time for each veg type.

In [ ]:
transition_slopes = site_climate_pheno.groupby("veg_type")[["transition_slope_rising", "transition_slope_falling"]].agg("median")

## Attach model predictions

Make sure we transform the rising date back to DOY

In [ ]:
# site_transitions_model = pd.merge(site_climate_pheno, pred_params, left_index=True, right_index=True)
site_transitions_model = site_climate_pheno.copy()
site_transitions_model["transition_slope_rising"] = site_transitions_model['veg_type'].map(transition_slopes["transition_slope_rising"])
site_transitions_model["transition_slope_falling"] = site_transitions_model['veg_type'].map(transition_slopes["transition_slope_falling"])

In [ ]:
site_transitions_model.to_parquet("phenocam_phenology_model.parquet")